In [1]:
%pip install imbalanced-learn

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.3.1 -> 25.1.1
[notice] To update, run: C:\Users\Ale\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.10_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [6]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report
from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.over_sampling import SMOTE
from ydata_profiling import ProfileReport

# 1. Cargar CSV
df = pd.read_csv("merged_dataset.csv")

# 2. Seleccionar columnas relevantes
model_cols = [
    'num_items',
    'total_order_value',
    'product_photos_qty',
    'product_description_lenght',
    'delivery_time_delta',
    'is_late',
    'product_category_name_english',
    'customer_state',
    'review_category'
]
model_df = df[model_cols].copy()

# 3. Mapear variable objetivo
model_df['review_category'] = model_df['review_category'].map({'Buena': 1, 'Mala': 0})

# 4. Crear y guardar reporte (ligero)
profile = ProfileReport(model_df, title="Reporte del Dataset", minimal=True)
profile.to_file("reporte_ligero.html")

# 5. Separar X e y
X = model_df.drop('review_category', axis=1)
y = model_df['review_category']

# 6. Preprocesamiento
numerical_cols = [
    'num_items',
    'total_order_value',
    'product_photos_qty',
    'product_description_lenght',
    'delivery_time_delta',
    'is_late'
]
categorical_cols = ['product_category_name_english', 'customer_state']

# Asegurar consistencia
X[categorical_cols] = X[categorical_cols].fillna('missing').astype(str)

preprocessor = ColumnTransformer([
    ('num', StandardScaler(), numerical_cols),
    ('cat', OneHotEncoder(drop='first', handle_unknown='ignore'), categorical_cols)
])

# 7. Crear pipeline con SMOTE y regresión logística
pipeline = ImbPipeline(steps=[
    ('preprocessor', preprocessor),
    ('smote', SMOTE(random_state=42)),  # Balanceo
    ('classifier', LogisticRegression(max_iter=500, random_state=42))  # No se necesita class_weight
])

# 8. Entrenamiento y evaluación
X_train, X_test, y_train, y_test = train_test_split(X, y, stratify=y, test_size=0.2, random_state=42)
pipeline.fit(X_train, y_train)

y_pred = pipeline.predict(X_test)
print(classification_report(y_test, y_pred))



Summarize dataset:   0%|          | 0/5 [00:00<?, ?it/s]

Generate report structure:   0%|          | 0/1 [00:00<?, ?it/s]

Render HTML:   0%|          | 0/1 [00:00<?, ?it/s]

Export report to file:   0%|          | 0/1 [00:00<?, ?it/s]

              precision    recall  f1-score   support

           0       0.46      0.43      0.44      4519
           1       0.83      0.85      0.84     15370

    accuracy                           0.75     19889
   macro avg       0.64      0.64      0.64     19889
weighted avg       0.75      0.75      0.75     19889



In [ ]:
print(model_df.isnull().sum())
print(model_df['review_category'].value_counts(normalize=True))


delivered                        0
num_items                        0
avg_freight_value                0
total_order_value                0
product_photos_qty               0
product_description_lenght       0
estimated_delivery_days          0
actual_delivery_days             0
delivery_time_delta              0
is_late                          0
product_category_name_english    0
customer_state                   0
seller_state                     0
order_status                     0
review_category                  0
dtype: int64
review_category
1    0.77278
0    0.22722
Name: proportion, dtype: float64


In [ ]:
print(model_df['review_category'].isnull().sum())
print(model_df['review_category'].value_counts(dropna=False))


0
review_category
1    76846
0    22595
Name: count, dtype: int64
